<a href="https://colab.research.google.com/github/Ledja22/vancouver-city-brain/blob/main/data_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Imports and Setup


In [40]:
import os, time, requests, zipfile, io
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from scipy.spatial import cKDTree
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

DATA_DIR = Path('../data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data directory: {DATA_DIR.resolve()}')

Data directory: /content/data


In [ ]:
def download(url: str, dest: Path, desc: str = '', chunk_mb: int = 4) -> Path:
    if dest.exists():
        print(f'✓ Already exists: {dest.name} ({dest.stat().st_size/1e6:.1f} MB) — skipping.')
        return dest

    print(f'⬇  {desc or url}')
    r = requests.get(url, stream=True, timeout=120,
                     headers={'User-Agent': 'citybrain-downloader/1.0'})
    r.raise_for_status()

    total = int(r.headers.get('content-length', 0))
    chunk = chunk_mb * 1024 * 1024

    with open(dest, 'wb') as f, tqdm(
        total=total, unit='B', unit_scale=True, desc=dest.name
    ) as bar:
        for data in r.iter_content(chunk_size=chunk):
            f.write(data)
            bar.update(len(data))

    print(f'   → {dest.stat().st_size/1e6:.1f} MB saved')
    return dest

###Pavement Condition Rating


In [ ]:
download(
    url  = 'https://opendata.vancouver.ca/api/explore/v2.1/catalog/datasets/pavement-condition-rating/exports/csv?lang=en&timezone=America%2FVancouver&use_labels=true&delimiter=%2C',
    dest = DATA_DIR / 'pavement_condition.csv',
    desc = 'Pavement Condition Rating'
)

⬇  Pavement Condition Rating


pavement_condition.csv: 0.00B [00:00, ?B/s]

   → 3.5 MB saved


PosixPath('../data/pavement_condition.csv')

###311 Service Requests (year by year)

In [ ]:
svc_dest = DATA_DIR / '311_service_requests.csv'

if svc_dest.exists():
    print(f'✓ Already exists: {svc_dest.name} — skipping.')
else:
    # Vancouver splits 311 data into two separate datasets at the 2022 boundary
    datasets = {
        '2009-2021': 'https://opendata.vancouver.ca/api/explore/v2.1/catalog/datasets/3-1-1-service-requests-2009-2021/exports/csv?lang=en&timezone=America%2FVancouver&use_labels=true&delimiter=%2C',
        '2022-present': 'https://opendata.vancouver.ca/api/explore/v2.1/catalog/datasets/3-1-1-service-requests/exports/csv?lang=en&timezone=America%2FVancouver&use_labels=true&delimiter=%2C',
    }

    frames = []
    for label, url in datasets.items():
        tmp = DATA_DIR / f'311_{label}.csv'
        download(url=url, dest=tmp, desc=f'311 Service Requests {label}')
        try:
            df = pd.read_csv(tmp, low_memory=False)
            print(f'   {label}: {len(df):,} rows, columns: {list(df.columns[:5])} ...')
            frames.append(df)
        except Exception as e:
            print(f'⚠ Could not parse {tmp.name}: {e}')
        time.sleep(2)

    if frames:
        combined = pd.concat(frames, ignore_index=True)
        # Keep only 2019 onwards to match our study window
        date_col = next((c for c in combined.columns if 'date' in c.lower()), None)
        if date_col:
            combined[date_col] = pd.to_datetime(combined[date_col], errors='coerce')
            combined = combined[combined[date_col].dt.year >= 2019]
            print(f'\n   Filtered to 2019+: {len(combined):,} rows')
        combined.to_csv(svc_dest, index=False)
        print(f'✓ Saved {len(combined):,} rows → {svc_dest}')
        for label in datasets:
            (DATA_DIR / f'311_{label}.csv').unlink(missing_ok=True)

⬇  311 Service Requests 2009-2021


311_2009-2021.csv: 0.00B [00:00, ?B/s]

   → 413.8 MB saved
   2009-2021: 2,083,091 rows, columns: ['Department', 'Service request type', 'Status', 'Closure reason', 'Service request open timestamp'] ...
⬇  311 Service Requests 2022-present


311_2022-present.csv: 0.00B [00:00, ?B/s]

   → 228.7 MB saved
   2022-present: 1,161,515 rows, columns: ['Department', 'Service request type', 'Status', 'Closure reason', 'Service request open timestamp'] ...

   Filtered to 2019+: 1,874,196 rows
✓ Saved 1,874,196 rows → ../data/311_service_requests.csv


In [58]:
import requests
import numpy as np
import time

def get_elevations_batch(lats, lons, batch_size=100):
    """
    Free elevation API — no key needed.
    Returns elevation in metres for each lat/lon pair.
    """
    elevations = []
    total = len(lats)

    for i in range(0, total, batch_size):
        batch_lats = lats[i:i+batch_size]
        batch_lons = lons[i:i+batch_size]
        locations  = '|'.join([f'{la},{lo}' for la, lo in zip(batch_lats, batch_lons)])

        try:
            r = requests.get(
                f'https://api.opentopodata.org/v1/srtm30m?locations={locations}',
                timeout=30
            )
            results = r.json()['results']
            elevations.extend([res['elevation'] for res in results])
        except Exception as e:
            print(f'Batch {i} failed: {e}')
            elevations.extend([None] * len(batch_lats))

        if i % 500 == 0:
            print(f'  {i}/{total} done...')
        time.sleep(1)   # API rate limit

    return elevations

# Get elevation for all 8,522 segments
print('Fetching elevation for all road segments (~2 mins)...')
lats = pci_joined['lat'].tolist()
lons = pci_joined['lon'].tolist()

elevations = get_elevations_batch(lats, lons)
pci_joined['elevation_m'] = elevations
pci_joined['elevation_m'] = pci_joined['elevation_m'].fillna(pci_joined['elevation_m'].median())

print(f'\n✓ Done')
print(f'Elevation range: {pci_joined["elevation_m"].min():.1f}m → {pci_joined["elevation_m"].max():.1f}m')
print(f'Mean elevation: {pci_joined["elevation_m"].mean():.1f}m')

Fetching elevation for all road segments (~2 mins)...
  0/8522 done...
  500/8522 done...
  1000/8522 done...
  1500/8522 done...
  2000/8522 done...
  2500/8522 done...
  3000/8522 done...
  3500/8522 done...
  4000/8522 done...
  4500/8522 done...
  5000/8522 done...
  5500/8522 done...
  6000/8522 done...
  6500/8522 done...
  7000/8522 done...
  7500/8522 done...
  8000/8522 done...
  8500/8522 done...

✓ Done
Elevation range: -1.0m → 127.0m
Mean elevation: 55.7m


In [60]:
from scipy.spatial import cKDTree

# For each segment, find its 5 nearest neighbours and compute elevation difference / distance
# That gives us slope estimate

coords = np.array(list(zip(pci_joined['lat'], pci_joined['lon'])))
elev   = pci_joined['elevation_m'].values

tree = cKDTree(coords)
distances, indices = tree.query(coords, k=6)  # k=6: itself + 5 neighbours

slopes = []
for i in range(len(coords)):
    neighbour_idx  = indices[i, 1:]   # skip itself
    neighbour_dist = distances[i, 1:]
    neighbour_elev = elev[neighbour_idx]

    # slope = max elevation change / distance (in degrees approx)
    elev_diff = np.abs(neighbour_elev - elev[i])
    # convert degree distance to metres (1 degree ≈ 111,000m)
    dist_m    = neighbour_dist * 111000
    slope_pct = np.max(elev_diff / (dist_m + 1e-8)) * 100
    slopes.append(slope_pct)

pci_joined['slope_pct'] = slopes

print(f'Slope range: {pci_joined["slope_pct"].min():.2f}% → {pci_joined["slope_pct"].max():.2f}%')
print(f'Mean slope:  {pci_joined["slope_pct"].mean():.2f}%')
print(f'\nTop 10 steepest segments:')
print(pci_joined[['Road Name', 'neighbourhood', 'elevation_m', 'slope_pct', 'PCI Rating']]
      .sort_values('slope_pct', ascending=False).head(10))

Slope range: 0.39% → 42.62%
Mean slope:  4.92%

Top 10 steepest segments:
              Road Name neighbourhood  elevation_m  slope_pct PCI Rating
5992       BROUGHTON ST      Downtown         17.0  42.624480       GOOD
7257          PENDER ST      Downtown         53.0  42.624480  VERY GOOD
1772        HASTINGS ST      Downtown         25.0  34.613343  VERY GOOD
5639          NICOLA ST      Downtown         38.0  33.847731  VERY GOOD
9000  COAL HARBOUR QUAY      Downtown          3.0  33.847731  VERY GOOD
7374          NICOLA ST      West End         26.0  33.691206       FAIR
5856         ALBERNI ST      West End         53.0  33.691206       POOR
6285        DUNSMUIR ST      Downtown         65.0  33.074426       FAIR
6328        HAMILTON ST      Downtown         30.0  33.074426       POOR
8019       BROUGHTON ST      Downtown         43.0  32.618605       GOOD


In [61]:
import matplotlib.pyplot as plt

# Check if elevation/slope correlate with PCI rating
print('=== Mean elevation by PCI Rating ===')
print(pci_joined.groupby('PCI Rating')['elevation_m'].mean().sort_values())

print('\n=== Mean slope by PCI Rating ===')
print(pci_joined.groupby('PCI Rating')['slope_pct'].mean().sort_values())

print('\n=== Mean complaints by PCI Rating ===')
tmp = pci_joined.merge(complaint_counts, on='segment_key', how='left')
tmp['total_complaints'] = tmp['total_complaints'].fillna(0)
print(tmp.groupby('PCI Rating')['total_complaints'].mean().sort_values(ascending=False))

# Correlation matrix
print('\n=== Correlation with risk_label ===')
check_cols = ['elevation_m', 'slope_pct', 'length_(m)']
for col in check_cols:
    corr = pci_joined[col].corr(pci_joined['risk_label'])
    print(f'  {col:20s}  r = {corr:.4f}')

=== Mean elevation by PCI Rating ===
PCI Rating
VERY GOOD    49.376301
GOOD         55.112420
POOR         57.060014
VERY POOR    58.136280
FAIR         59.024168
Name: elevation_m, dtype: float64

=== Mean slope by PCI Rating ===
PCI Rating
POOR         4.749347
VERY POOR    4.812911
FAIR         4.911653
GOOD         5.023029
VERY GOOD    5.079207
Name: slope_pct, dtype: float64

=== Mean complaints by PCI Rating ===
PCI Rating
VERY POOR    6.195826
POOR         4.892533
FAIR         4.281955
GOOD         4.023019
VERY GOOD    3.818497
Name: total_complaints, dtype: float64

=== Correlation with risk_label ===
  elevation_m           r = 0.0822
  slope_pct             r = -0.0367
  length_(m)            r = 0.0522


In [63]:
# Re-parse year column on road_complaints
road_complaints['date'] = pd.to_datetime(
    road_complaints['Service request open timestamp'], utc=True, errors='coerce'
)
road_complaints['year'] = road_complaints['date'].dt.year
print(f'road_complaints rows: {len(road_complaints):,}')
print(f'year range: {road_complaints["year"].min()} → {road_complaints["year"].max()}')

road_complaints rows: 38,792
year range: 2019 → 2025


In [64]:
# Feature 1 — 2019-only complaints (causally before the survey)
road_complaints_2019 = road_complaints[
    road_complaints['year'] == 2019
].copy()

complaint_coords_2019 = np.array(list(zip(
    road_complaints_2019['Latitude'],
    road_complaints_2019['Longitude']
)))

distances_2019, indices_2019 = tree.query(complaint_coords_2019, k=1)
road_complaints_2019 = road_complaints_2019.copy()
road_complaints_2019['pci_idx'] = indices_2019
road_complaints_2019['dist']    = distances_2019
road_complaints_2019 = road_complaints_2019[road_complaints_2019['dist'] <= 0.001]
road_complaints_2019['segment_key'] = road_complaints_2019['pci_idx'].map(
    lambda i: pci_joined.iloc[i]['segment_key']
)
counts_2019 = road_complaints_2019.groupby('segment_key').size().reset_index(
    name='complaints_2019'
)

# Feature 2 — neighbourhood risk profile
# % of roads in the same neighbourhood that are POOR or VERY POOR
hood_risk = pci_joined.groupby('neighbourhood').apply(
    lambda x: (x['risk_label'] == 2).sum() / len(x)
).reset_index(name='neighbourhood_high_risk_pct')

# Feature 3 — segment density (how many other segments within 200m)
# Dense areas = more traffic stress
tree_all   = cKDTree(np.array(list(zip(pci_joined['lat'], pci_joined['lon']))))
neighbour_counts = tree_all.query_ball_point(
    np.array(list(zip(pci_joined['lat'], pci_joined['lon']))),
    r=0.002   # ~200m
)
pci_joined['segment_density'] = [len(n) - 1 for n in neighbour_counts]

print('2019 complaint counts:')
print(counts_2019.describe())

print('\nNeighbourhood high-risk %:')
print(hood_risk.sort_values('neighbourhood_high_risk_pct', ascending=False))

print('\nSegment density stats:')
print(pci_joined['segment_density'].describe())

2019 complaint counts:
       complaints_2019
count      2874.000000
mean          1.755741
std           1.336068
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          15.000000

Neighbourhood high-risk %:
               neighbourhood  neighbourhood_high_risk_pct
15               Shaughnessy                     0.617470
18                    Sunset                     0.473684
21           West Point Grey                     0.456790
0              Arbutus Ridge                     0.425170
14                Riley Park                     0.422122
6   Kensington-Cedar Cottage                     0.386067
7                 Kerrisdale                     0.376582
13       Renfrew-Collingwood                     0.372163
16              South Cambie                     0.371795
5           Hastings-Sunrise                     0.371383
19       Victoria-Fraserview                     0.360262
4         Grandview-Woodland                  

/tmp/ipykernel_3651/729915874.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hood_risk = pci_joined.groupby('neighbourhood').apply(


###Weather, Environment Canada (year by year)


In [ ]:
import requests
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
wx_dest = DATA_DIR / 'weather_vancouver.csv'

# Open-Meteo historical weather API — no API key needed
# Vancouver coordinates: 49.2497° N, 123.1193° W
url = (
    "https://archive-api.open-meteo.com/v1/archive"
    "?latitude=49.2497&longitude=-123.1193"
    "&start_date=2019-01-01&end_date=2025-12-31"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "precipitation_sum,snowfall_sum,snow_depth_mean"
    "&timezone=America%2FVancouver"
)

print('Fetching weather from Open-Meteo...')
r = requests.get(url, timeout=60)
r.raise_for_status()
data = r.json()

df = pd.DataFrame(data['daily'])
df.rename(columns={
    'time':                  'date',
    'temperature_2m_max':    'max_temp_c',
    'temperature_2m_min':    'min_temp_c',
    'temperature_2m_mean':   'mean_temp_c',
    'precipitation_sum':     'total_precip_mm',
    'snowfall_sum':          'snow_on_ground_cm',
}, inplace=True)

df['date'] = pd.to_datetime(df['date'])

# Add freeze-thaw flag — exactly what preprocessing.py expects
df['freeze_thaw_flag'] = (
    (df['min_temp_c'] < 0) & (df['max_temp_c'] > 0)
).astype(int)

df.to_csv(wx_dest, index=False)
print(f'✓ {len(df):,} daily rows saved → {wx_dest}')
print(f'Columns: {list(df.columns)}')
print(f'Missing values:\n{df.isnull().sum()}')

Fetching weather from Open-Meteo...
✓ 2,557 daily rows saved → ../data/weather_vancouver.csv
Columns: ['date', 'max_temp_c', 'min_temp_c', 'mean_temp_c', 'total_precip_mm', 'snow_on_ground_cm', 'snow_depth_mean', 'freeze_thaw_flag']
Missing values:
date                 0
max_temp_c           0
min_temp_c           0
mean_temp_c          0
total_precip_mm      0
snow_on_ground_cm    0
snow_depth_mean      0
freeze_thaw_flag     0
dtype: int64


In [ ]:
import shutil
from pathlib import Path

src = Path('../data')
dst = Path('../vancouver-city-brain/data')
dst.mkdir(parents=True, exist_ok=True)

# Force copy ALL files fresh
for f in src.iterdir():
    if f.is_file():
        shutil.copy(f, dst / f.name)
        print(f'✓ {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

# Verify weather is fixed
wx = pd.read_csv(dst / 'weather_vancouver.csv', nrows=2)
print(f'\nWeather columns: {list(wx.columns)}')
print('✅ All good!' if 'max_temp_c' in wx.columns else '❌ Weather still broken')

✓ infrastructure_condition.csv  (1.3 MB)
✓ local-streets.geojson  (0.1 MB)
✓ pavement_condition.csv  (3.5 MB)
✓ weather_vancouver.csv  (0.1 MB)
✓ 311_service_requests.csv  (367.5 MB)

Weather columns: ['date', 'max_temp_c', 'min_temp_c', 'mean_temp_c', 'total_precip_mm', 'snow_on_ground_cm', 'snow_depth_mean', 'freeze_thaw_flag']
✅ All good!


### Infrastructure Condition

In [ ]:
download(
    url  = 'https://opendata.vancouver.ca/api/explore/v2.1/catalog/datasets/pavement-condition-rating-major-road-network/exports/csv?lang=en&timezone=America%2FVancouver&use_labels=true&delimiter=%2C',
    dest = DATA_DIR / 'infrastructure_condition.csv',
    desc = 'Pavement Condition Rating — Major Road Network (infrastructure proxy)'
)

✓ Already exists: infrastructure_condition.csv (1.3 MB) — skipping.


PosixPath('../data/infrastructure_condition.csv')

###GeoJSON road network (demo map)

In [ ]:
download(
    url  = 'https://opendata.vancouver.ca/api/explore/v2.1/catalog/datasets/local-area-boundary/exports/geojson?lang=en&timezone=America%2FVancouver',
    dest = DATA_DIR / 'local-streets.geojson',
    desc = 'Road network GeoJSON'
)

✓ Already exists: local-streets.geojson (0.1 MB) — skipping.


PosixPath('../data/local-streets.geojson')

In [ ]:
##just to make sure printing out all the right files here again
files = [
    'pavement_condition.csv',
    '311_service_requests.csv',
    'weather_vancouver.csv',
    'infrastructure_condition.csv',
]

print('=' * 60)
for fname in files:
    path = DATA_DIR / fname
    if not path.exists():
        print(f'✗  MISSING   {fname}')
        continue
    df = pd.read_csv(path, nrows=3, low_memory=False)
    print(f'✓  {fname}  ({path.stat().st_size/1e6:.1f} MB, {len(df.columns)} cols)')
    for c in df.columns:
        print(f'     {c}')
    print()

✓  pavement_condition.csv  (3.5 MB, 8 cols)
     Year
     Road Name
     From Street
     To Street
     length_(m)
     PCI Rating
     Geom
     geo_point_2d

✓  311_service_requests.csv  (367.5 MB, 13 cols)
     Department
     Service request type
     Status
     Closure reason
     Service request open timestamp
     Service request close date
     Last modified timestamp
     Address
     Local area
     Channel
     Latitude
     Longitude
     geom

✓  weather_vancouver.csv  (1.1 MB, 1 cols)
     	<ul id="wb-tphp">

✓  infrastructure_condition.csv  (1.3 MB, 8 cols)
     Year
     Road Name
     From Street
     To Street
     length_(m)
     PCI Rating
     Geom
     geo_point_2d



In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../vancouver-city-brain/data')

# Check PCI Rating values
pci = pd.read_csv(DATA_DIR / 'pavement_condition.csv')
print('=== PCI Rating unique values ===')
print(pci['PCI Rating'].value_counts())

print('\n=== PCI Year values ===')
print(pci['Year'].value_counts())

# Check 311 local areas
svc = pd.read_csv(DATA_DIR / '311_service_requests.csv', low_memory=False)
print('\n=== 311 Local area unique values ===')
print(svc['Local area'].value_counts().head(20))

print('\n=== 311 date range ===')
svc['date'] = pd.to_datetime(svc['Service request open timestamp'], errors='coerce')
print(f'From: {svc["date"].min()}')
print(f'To:   {svc["date"].max()}')

# Check neighbourhood overlap between PCI and 311
infra = pd.read_csv(DATA_DIR / 'infrastructure_condition.csv')
print('\n=== Does PCI have a neighbourhood column? ===')
print(pci.columns.tolist())

=== PCI Rating unique values ===
PCI Rating
GOOD         1876
FAIR         1871
VERY GOOD    1758
VERY POOR    1641
POOR         1440
NO DATA       430
Name: count, dtype: int64

=== PCI Year values ===
Year
2020    9016
Name: count, dtype: int64

=== 311 Local area unique values ===
Local area
Downtown                    186579
Kensington-Cedar Cottage    117985
Kitsilano                   100532
Renfrew-Collingwood          95610
Grandview-Woodland           89244
Mount Pleasant               87296
Hastings-Sunrise             83092
West End                     76155
Sunset                       72654
Strathcona                   68154
Riley Park                   66197
Fairview                     65451
Victoria-Fraserview          53559
Dunbar-Southlands            53487
Marpole                      50736
Killarney                    43647
West Point Grey              38996
Kerrisdale                   33902
Arbutus Ridge                30395
Oakridge                     29856
Name

/tmp/ipykernel_3651/2744563265.py:20: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  svc['date'] = pd.to_datetime(svc['Service request open timestamp'], errors='coerce')


From: 2010-12-14 16:58:47-08:00
To:   2026-04-08 21:26:00-07:00

=== Does PCI have a neighbourhood column? ===
['Year', 'Road Name', 'From Street', 'To Street', 'length_(m)', 'PCI Rating', 'Geom', 'geo_point_2d']


In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point

DATA_DIR = Path('../vancouver-city-brain/data')

# Load PCI and drop NO DATA
pci = pd.read_csv(DATA_DIR / 'pavement_condition.csv')
pci = pci[pci['PCI Rating'] != 'NO DATA'].copy()
print(f'PCI rows after dropping NO DATA: {len(pci)}')

# Parse lat/lon from geo_point_2d  (format: "49.1234, -123.1234")
pci[['lat', 'lon']] = pci['geo_point_2d'].str.split(',', expand=True).astype(float)

# Convert to GeoDataFrame
pci_gdf = gpd.GeoDataFrame(
    pci,
    geometry=gpd.points_from_xy(pci['lon'], pci['lat']),
    crs='EPSG:4326'
)

# Load neighbourhood boundaries
neighbourhoods = gpd.read_file(DATA_DIR / 'local-streets.geojson')
print(f'Neighbourhood columns: {neighbourhoods.columns.tolist()}')
print(f'Neighbourhoods: {neighbourhoods.shape[0]} areas')

# Spatial join — assign each road segment to a neighbourhood
pci_joined = gpd.sjoin(pci_gdf, neighbourhoods, how='left', predicate='within')
print(f'\nColumns after join: {pci_joined.columns.tolist()}')
print(f'Segments with neighbourhood assigned: {pci_joined["name"].notna().sum()}')
print(f'Segments without neighbourhood:       {pci_joined["name"].isna().sum()}')

PCI rows after dropping NO DATA: 8586
Neighbourhood columns: ['name', 'geo_point_2d', 'geometry']
Neighbourhoods: 22 areas

Columns after join: ['Year', 'Road Name', 'From Street', 'To Street', 'length_(m)', 'PCI Rating', 'Geom', 'geo_point_2d_left', 'lat', 'lon', 'geometry', 'index_right', 'name', 'geo_point_2d_right']
Segments with neighbourhood assigned: 8522
Segments without neighbourhood:       71


In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
from shapely.geometry import Point

DATA_DIR = Path('../vancouver-city-brain/data')

# Load only road-related complaints with coordinates
svc = pd.read_csv(DATA_DIR / '311_service_requests.csv', low_memory=False)
road_complaints = svc[
    svc['Service request type'].isin(['Pothole Case', 'Street Repair Case']) &
    svc['Latitude'].notna() &
    svc['Longitude'].notna()
].copy()

print(f'Road complaints with coordinates: {len(road_complaints):,}')

# Parse date
road_complaints['date'] = pd.to_datetime(
    road_complaints['Service request open timestamp'], utc=True, errors='coerce'
)
road_complaints['year'] = road_complaints['date'].dt.year

print(f'Date range: {road_complaints["date"].min()} → {road_complaints["date"].max()}')
print(f'\nComplaints per year:')
print(road_complaints['year'].value_counts().sort_index())

Road complaints with coordinates: 44,250
Date range: 2011-09-13 21:39:21+00:00 → 2026-04-08 00:52:45+00:00

Complaints per year:
year
2011       1
2014       9
2015      11
2016      18
2017     105
2018     451
2019    5544
2020    6286
2021    5958
2022    7599
2023    5720
2024    5395
2025    5552
2026    1601
Name: count, dtype: int64


In [ ]:
from scipy.spatial import cKDTree

# Load PCI with coordinates
pci = pd.read_csv(DATA_DIR / 'pavement_condition.csv')
pci = pci[pci['PCI Rating'] != 'NO DATA'].copy()
pci[['lat', 'lon']] = pci['geo_point_2d'].str.split(',', expand=True).astype(float)

# Filter complaints to 2019-2025 only
road_complaints = road_complaints[
    road_complaints['year'].between(2019, 2025)
].copy()
print(f'Complaints 2019-2025: {len(road_complaints):,}')

# Build KD-tree from PCI segment coordinates
pci_coords = np.array(list(zip(pci['lat'], pci['lon'])))
complaint_coords = np.array(list(zip(
    road_complaints['Latitude'],
    road_complaints['Longitude']
)))

tree = cKDTree(pci_coords)
distances, indices = tree.query(complaint_coords, k=1)

# Only keep complaints within 100 metres (approx 0.001 degrees)
MAX_DIST = 0.001
road_complaints['pci_idx'] = indices
road_complaints['dist']    = distances
road_complaints = road_complaints[road_complaints['dist'] <= MAX_DIST]
road_complaints['segment_key'] = road_complaints['pci_idx'].map(
    lambda i: f"{pci.iloc[i]['Road Name']}_{pci.iloc[i]['From Street']}"
)

print(f'Complaints matched to a segment (within 100m): {len(road_complaints):,}')

# Count complaints per segment
complaint_counts = road_complaints.groupby('segment_key').size().reset_index(
    name='total_complaints_2019_2025'
)
print(f'Unique segments with at least 1 complaint: {len(complaint_counts):,}')
print(f'\nTop 10 most complained segments:')
print(complaint_counts.sort_values('total_complaints_2019_2025', ascending=False).head(10))

Complaints 2019-2025: 42,054
Complaints matched to a segment (within 100m): 38,861
Unique segments with at least 1 complaint: 6,980

Top 10 most complained segments:
                 segment_key  total_complaints_2019_2025
3138  COMMERCIAL DR_BROADWAY                          85
2904   CARRALL ST_CORDOVA ST                          71
3124   COMMERCIAL DR_10TH AV                          60
1928       62ND AV_BORDEN ST                          59
4373         KERR ST_49TH AV                          58
5803     ROBSON ST_DENMAN ST                          56
2802    CAMBIE ST_CORDOVA ST                          50
6202    ST GEORGE ST_55TH AV                          50
2040      6TH AV_KEITH DRIVE                          48
2770  BURRARD ST_CORNWALL AV                          48


### Load & clean PCI


In [41]:
pci = pd.read_csv(DATA_DIR / 'pavement_condition.csv')
pci = pci[pci['PCI Rating'] != 'NO DATA'].copy()
pci[['lat', 'lon']] = pci['geo_point_2d'].str.split(',', expand=True).astype(float)

# Create segment key for joining complaints
pci['segment_key'] = pci['Road Name'] + '_' + pci['From Street']

# Encode label
label_map = {
    'VERY GOOD': 0,
    'GOOD':      0,
    'FAIR':      1,
    'POOR':      2,
    'VERY POOR': 2,
}
pci['risk_label'] = pci['PCI Rating'].map(label_map)

print(f'Segments: {len(pci):,}')
print(f'Label distribution:\n{pci["risk_label"].value_counts().sort_index()}')

Segments: 8,586
Label distribution:
risk_label
0    3634
1    1871
2    3081
Name: count, dtype: int64


### Assign neighbourhood via spatial join


In [42]:
neighbourhoods = gpd.read_file(DATA_DIR / 'local-streets.geojson')

pci_gdf = gpd.GeoDataFrame(
    pci,
    geometry=gpd.points_from_xy(pci['lon'], pci['lat']),
    crs='EPSG:4326'
)

pci_joined = gpd.sjoin(pci_gdf, neighbourhoods[['name', 'geometry']],
                        how='left', predicate='within')

# Drop the 71 unmatched segments
pci_joined = pci_joined[pci_joined['name'].notna()].copy()
pci_joined = pci_joined.rename(columns={'name': 'neighbourhood'})

print(f'Segments after neighbourhood join: {len(pci_joined):,}')
print(f'Neighbourhoods: {pci_joined["neighbourhood"].nunique()}')

Segments after neighbourhood join: 8,522
Neighbourhoods: 22


### Build per-segment complaint counts

In [43]:
svc = pd.read_csv(DATA_DIR / '311_service_requests.csv', low_memory=False)

road_complaints = svc[
    svc['Service request type'].isin(['Pothole Case', 'Street Repair Case']) &
    svc['Latitude'].notna() &
    svc['Longitude'].notna()
].copy()

road_complaints['date'] = pd.to_datetime(
    road_complaints['Service request open timestamp'], utc=True, errors='coerce'
)
road_complaints = road_complaints[
    road_complaints['date'].dt.year.between(2019, 2025)
].copy()

# Snap to nearest PCI segment
pci_coords       = np.array(list(zip(pci_joined['lat'], pci_joined['lon'])))
complaint_coords = np.array(list(zip(road_complaints['Latitude'],
                                      road_complaints['Longitude'])))

tree = cKDTree(pci_coords)
distances, indices = tree.query(complaint_coords, k=1)

road_complaints['pci_idx'] = indices
road_complaints['dist']    = distances
road_complaints = road_complaints[road_complaints['dist'] <= 0.001]
road_complaints['segment_key'] = road_complaints['pci_idx'].map(
    lambda i: pci_joined.iloc[i]['segment_key']
)

complaint_counts = road_complaints.groupby('segment_key').size().reset_index(
    name='total_complaints'
)

print(f'Segments with complaints: {len(complaint_counts):,}')

Segments with complaints: 6,957


### Build weather aggregates

In [44]:
wx = pd.read_csv(DATA_DIR / 'weather_vancouver.csv')
wx['date'] = pd.to_datetime(wx['date'])

# City-wide annual aggregates (2019-2025)
wx_agg = wx.groupby(wx['date'].dt.year).agg(
    freeze_thaw_count  = ('freeze_thaw_flag', 'sum'),
    total_precip_mm    = ('total_precip_mm',  'sum'),
    avg_min_temp_c     = ('min_temp_c',       'mean'),
    avg_max_temp_c     = ('max_temp_c',       'mean'),
).reset_index().rename(columns={'date': 'year'})

# Single weather vector = mean across 2019-2025
wx_features = {
    'freeze_thaw_count': wx_agg['freeze_thaw_count'].mean(),
    'total_precip_mm':   wx_agg['total_precip_mm'].mean(),
    'avg_min_temp_c':    wx_agg['avg_min_temp_c'].mean(),
    'avg_max_temp_c':    wx_agg['avg_max_temp_c'].mean(),
}
print('Weather features (2019-2025 mean):')
for k, v in wx_features.items():
    print(f'  {k}: {v:.2f}')

Weather features (2019-2025 mean):
  freeze_thaw_count: 28.71
  total_precip_mm: 1724.33
  avg_min_temp_c: 7.36
  avg_max_temp_c: 13.16


### Master join

In [45]:
# Join complaints onto PCI
master = pci_joined.merge(complaint_counts, on='segment_key', how='left')
master['total_complaints'] = master['total_complaints'].fillna(0)
master['has_complaint']    = (master['total_complaints'] > 0).astype(float)
master['complaints_per_m'] = master['total_complaints'] / (master['length_(m)'] + 1e-8)

# Weather is city-wide — broadcast to all segments
for k, v in wx_features.items():
    master[k] = v

print(f'Master dataset: {len(master):,} segments')
print(f'Label distribution:\n{master["risk_label"].value_counts().sort_index()}')
print(f'\nFeature preview:')
print(master[['Road Name', 'neighbourhood', 'length_(m)',
              'total_complaints', 'freeze_thaw_count', 'risk_label']].head())

Master dataset: 8,522 segments
Label distribution:
risk_label
0    3598
1    1862
2    3062
Name: count, dtype: int64

Feature preview:
  Road Name             neighbourhood  length_(m)  total_complaints  \
0   47TH AV                 Killarney          89               2.0   
1   62ND AV                Kerrisdale         206               2.0   
2   29TH AV  Kensington-Cedar Cottage         101               3.0   
3   31ST AV  Kensington-Cedar Cottage          93               2.0   
4   53RD AV       Victoria-Fraserview         100               6.0   

   freeze_thaw_count  risk_label  
0          28.714286           0  
1          28.714286           2  
2          28.714286           2  
3          28.714286           2  
4          28.714286           0  


### Encode categoricals + scale


In [46]:
le_neighbourhood = LabelEncoder()
le_roadtype      = LabelEncoder()

master['neighbourhood_enc'] = le_neighbourhood.fit_transform(master['neighbourhood'])
master['road_name_enc']     = le_roadtype.fit_transform(master['Road Name'])

# Branch 1: Road features
road_cols = ['length_(m)', 'neighbourhood_enc', 'road_name_enc']

# Branch 2: Weather features
weather_cols = ['freeze_thaw_count', 'total_precip_mm',
                'avg_min_temp_c', 'avg_max_temp_c']

# Branch 3: Complaint features
complaint_cols = ['total_complaints', 'has_complaint', 'complaints_per_m']

X_road      = master[road_cols].values.astype(np.float32)
X_weather   = master[weather_cols].values.astype(np.float32)
X_complaint = master[complaint_cols].values.astype(np.float32)
y           = master['risk_label'].values.astype(np.int64)

scaler_road      = StandardScaler()
scaler_weather   = StandardScaler()
scaler_complaint = StandardScaler()

X_road      = scaler_road.fit_transform(X_road)
X_weather   = scaler_weather.fit_transform(X_weather)
X_complaint = scaler_complaint.fit_transform(X_complaint)

print(f'Road features shape:      {X_road.shape}')
print(f'Weather features shape:   {X_weather.shape}')
print(f'Complaint features shape: {X_complaint.shape}')
print(f'Labels shape:             {y.shape}')
print(f'\nClass counts: low={np.sum(y==0):,}  medium={np.sum(y==1):,}  high={np.sum(y==2):,}')

Road features shape:      (8522, 3)
Weather features shape:   (8522, 4)
Complaint features shape: (8522, 3)
Labels shape:             (8522,)

Class counts: low=3,598  medium=1,862  high=3,062


###  Dataset class + dataloaders

In [47]:
class RoadRiskDataset(Dataset):
    def __init__(self, road, weather, complaint, labels):
        self.road      = torch.tensor(road,      dtype=torch.float32)
        self.weather   = torch.tensor(weather,   dtype=torch.float32)
        self.complaint = torch.tensor(complaint, dtype=torch.float32)
        self.labels    = torch.tensor(labels,    dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (self.road[idx], self.weather[idx],
                self.complaint[idx], self.labels[idx])


def make_dataloaders(X_road, X_weather, X_complaint, y,
                     batch_size=64, seed=42):
    N   = len(y)
    idx = np.arange(N)

    idx_train, idx_temp = train_test_split(
        idx, test_size=0.30, stratify=y, random_state=seed
    )
    idx_val, idx_test = train_test_split(
        idx_temp, test_size=0.50, stratify=y[idx_temp], random_state=seed
    )

    def _subset(i):
        return RoadRiskDataset(X_road[i], X_weather[i],
                                X_complaint[i], y[i])

    train_loader = DataLoader(_subset(idx_train), batch_size=batch_size,
                               shuffle=True)
    val_loader   = DataLoader(_subset(idx_val),   batch_size=batch_size,
                               shuffle=False)
    test_loader  = DataLoader(_subset(idx_test),  batch_size=batch_size,
                               shuffle=False)

    print(f'Train: {len(idx_train):,}  Val: {len(idx_val):,}  Test: {len(idx_test):,}')
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_dataloaders(
    X_road, X_weather, X_complaint, y
)

Train: 5,965  Val: 1,278  Test: 1,279


### Three branch models

### Fusion model

In [49]:
class CityBrainFusion(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.road_branch      = RoadMLP(embed_dim=32, dropout=dropout)
        self.weather_branch   = WeatherMLP(embed_dim=32, dropout=dropout)
        self.complaint_branch = ComplaintMLP(embed_dim=32, dropout=dropout)

        self.fusion_head = nn.Sequential(
            nn.Linear(96, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    def forward(self, road, weather, complaint):
        e1 = self.road_branch(road)
        e2 = self.weather_branch(weather)
        e3 = self.complaint_branch(complaint)
        return self.fusion_head(torch.cat([e1, e2, e3], dim=-1))

print('✓ Fusion model defined')

✓ Fusion model defined


### Training loop

In [50]:
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3, mode='fusion'):
    # Class-weighted loss for imbalance
    counts  = np.bincount(y, minlength=3)
    weights = torch.tensor(1.0 / (counts + 1e-8), dtype=torch.float32).to(DEVICE)
    weights /= weights.sum()
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_f1, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        for batch in train_loader:
            road, weather, complaint, labels = [b.to(DEVICE) for b in batch]
            optimizer.zero_grad()
            logits = _forward(model, mode, road, weather, complaint)
            loss   = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        # Validate
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in val_loader:
                road, weather, complaint, labels = [b.to(DEVICE) for b in batch]
                logits = _forward(model, mode, road, weather, complaint)
                preds.extend(logits.argmax(1).cpu().numpy())
                trues.extend(labels.cpu().numpy())

        val_f1 = f1_score(trues, preds, average='macro', zero_division=0)

        if val_f1 > best_f1:
            best_f1   = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0:
            print(f'Epoch {epoch:03d} | val_macro_f1={val_f1:.4f} | best={best_f1:.4f}')

    model.load_state_dict(best_state)
    return model, best_f1


def _forward(model, mode, road, weather, complaint):
    if mode == 'road':      return model(road)
    if mode == 'weather':   return model(weather)
    if mode == 'complaint': return model(complaint)
    return model(road, weather, complaint)


def evaluate(model, loader, mode='fusion'):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            road, weather, complaint, labels = [b.to(DEVICE) for b in batch]
            logits = _forward(model, mode, road, weather, complaint)
            preds.extend(logits.argmax(1).cpu().numpy())
            trues.extend(labels.cpu().numpy())
    print(classification_report(trues, preds,
          target_names=['low', 'medium', 'high'], digits=4))
    return f1_score(trues, preds, average='macro', zero_division=0)

print('✓ Training functions defined')

Device: cpu
✓ Training functions defined


In [65]:
# ── Rebuild master with all new features ─────────────────────────────────────

master = pci_joined.copy()

# Join 2019 complaints
master = master.merge(counts_2019, on='segment_key', how='left')
master['complaints_2019'] = master['complaints_2019'].fillna(0)

# Join all-years complaints
master = master.merge(complaint_counts, on='segment_key', how='left')
master['total_complaints'] = master['total_complaints'].fillna(0)

# Join neighbourhood risk profile
master = master.merge(hood_risk, on='neighbourhood', how='left')

# Derived features
master['has_complaint_2019']  = (master['complaints_2019'] > 0).astype(float)
master['complaints_per_m']    = master['total_complaints'] / (master['length_(m)'] + 1e-8)
master['complaint_x_density'] = master['total_complaints'] * master['segment_density']

# Encode categoricals
le_neighbourhood = LabelEncoder()
le_roadname      = LabelEncoder()
master['neighbourhood_enc'] = le_neighbourhood.fit_transform(master['neighbourhood'])
master['road_name_enc']     = le_roadname.fit_transform(master['Road Name'])

# ── Three branch feature sets ─────────────────────────────────────────────────

# Branch 1 — Road attributes
road_cols = [
    'length_(m)',
    'neighbourhood_enc',
    'segment_density',
    'neighbourhood_high_risk_pct',   # strongest new feature
]

# Branch 2 — Weather (city-wide)
weather_cols = [
    'freeze_thaw_count',
    'total_precip_mm',
    'avg_min_temp_c',
    'avg_max_temp_c',
]

# Branch 3 — Complaint signals
complaint_cols = [
    'complaints_2019',               # causally before survey
    'has_complaint_2019',
    'total_complaints',
    'complaints_per_m',
    'complaint_x_density',
]

# Add weather to master
for k, v in wx_features.items():
    master[k] = v

y = master['risk_label'].values.astype(np.int64)

X_road      = StandardScaler().fit_transform(
    master[road_cols].values.astype(np.float32))
X_weather   = StandardScaler().fit_transform(
    master[weather_cols].values.astype(np.float32))
X_complaint = StandardScaler().fit_transform(
    master[complaint_cols].values.astype(np.float32))

print(f'Road features:      {X_road.shape}')
print(f'Weather features:   {X_weather.shape}')
print(f'Complaint features: {X_complaint.shape}')
print(f'Labels:             {y.shape}')
print(f'\nClass counts: low={np.sum(y==0):,}  medium={np.sum(y==1):,}  high={np.sum(y==2):,}')

# Rebuild dataloaders
train_loader, val_loader, test_loader = make_dataloaders(
    X_road, X_weather, X_complaint, y
)
print('\n✓ Ready to train')

Road features:      (8522, 4)
Weather features:   (8522, 4)
Complaint features: (8522, 5)
Labels:             (8522,)

Class counts: low=3,598  medium=1,862  high=3,062
Train: 5,965  Val: 1,278  Test: 1,279

✓ Ready to train


In [68]:
# ── Model definitions with updated input dims ─────────────────────────────────

class RoadMLP(nn.Module):
    def __init__(self, input_dim=4, embed_dim=32, dropout=0.3, standalone=False):
        super().__init__()
        self.standalone = standalone
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, embed_dim),
        )
        if standalone: self.classifier = nn.Linear(embed_dim, 3)
    def forward(self, x):
        emb = self.net(x)
        return self.classifier(emb) if self.standalone else emb

class WeatherMLP(nn.Module):
    def __init__(self, input_dim=4, embed_dim=32, dropout=0.3, standalone=False):
        super().__init__()
        self.standalone = standalone
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, embed_dim),
        )
        if standalone: self.classifier = nn.Linear(embed_dim, 3)
    def forward(self, x):
        emb = self.net(x)
        return self.classifier(emb) if self.standalone else emb

class ComplaintMLP(nn.Module):
    def __init__(self, input_dim=5, embed_dim=32, dropout=0.3, standalone=False):
        super().__init__()
        self.standalone = standalone
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, embed_dim),
        )
        if standalone: self.classifier = nn.Linear(embed_dim, 3)
    def forward(self, x):
        emb = self.net(x)
        return self.classifier(emb) if self.standalone else emb

class CityBrainFusion(nn.Module):
    def __init__(self, road_dim=4, weather_dim=4, complaint_dim=5, dropout=0.3):
        super().__init__()
        self.road_branch      = RoadMLP(input_dim=road_dim,      embed_dim=32, dropout=dropout)
        self.weather_branch   = WeatherMLP(input_dim=weather_dim,  embed_dim=32, dropout=dropout)
        self.complaint_branch = ComplaintMLP(input_dim=complaint_dim, embed_dim=32, dropout=dropout)
        self.fusion_head = nn.Sequential(
            nn.Linear(96, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 3)
        )
    def forward(self, road, weather, complaint):
        e1 = self.road_branch(road)
        e2 = self.weather_branch(weather)
        e3 = self.complaint_branch(complaint)
        return self.fusion_head(torch.cat([e1, e2, e3], dim=-1))

# ── Train all four models ─────────────────────────────────────────────────────

results = {}
EPOCHS  = 100

for mode, model in [
    ('road',      RoadMLP(input_dim=4, standalone=True).to(DEVICE)),
    ('weather',   WeatherMLP(input_dim=4, standalone=True).to(DEVICE)),
    ('complaint', ComplaintMLP(input_dim=5, standalone=True).to(DEVICE)),
    ('fusion',    CityBrainFusion(road_dim=4, weather_dim=4, complaint_dim=5).to(DEVICE)),
]:
    print(f'\n{"="*50}')
    print(f'Training {mode}...')
    model, best_f1 = train_model(model, train_loader, val_loader,
                                  epochs=EPOCHS, lr=1e-3, mode=mode)
    print(f'\n--- {mode} TEST RESULTS ---')
    test_f1 = evaluate(model, test_loader, mode=mode)
    results[mode] = test_f1

# ── Summary ───────────────────────────────────────────────────────────────────

print(f'\n{"="*50}')
print('RESULTS SUMMARY')
print(f'{"="*50}')
best_baseline = max(results['road'], results['weather'], results['complaint'])
for name, f1 in results.items():
    print(f'{name:<12} macro F1 = {f1:.4f}')
print(f'\nFusion improvement over best baseline: {results["fusion"] - best_baseline:.4f}')
print(f'Target: 0.75  |  Gap: {0.75 - results["fusion"]:.4f}')


Training road...
Epoch 010 | val_macro_f1=0.3993 | best=0.4134
Epoch 020 | val_macro_f1=0.3916 | best=0.4249
Epoch 030 | val_macro_f1=0.4101 | best=0.4249
Epoch 040 | val_macro_f1=0.4003 | best=0.4249
Epoch 050 | val_macro_f1=0.4174 | best=0.4249
Epoch 060 | val_macro_f1=0.4043 | best=0.4249
Epoch 070 | val_macro_f1=0.4083 | best=0.4249
Epoch 080 | val_macro_f1=0.4037 | best=0.4249
Epoch 090 | val_macro_f1=0.4068 | best=0.4249
Epoch 100 | val_macro_f1=0.4023 | best=0.4249

--- road TEST RESULTS ---
              precision    recall  f1-score   support

         low     0.4703    0.5574    0.5102       540
      medium     0.2195    0.2258    0.2226       279
        high     0.4318    0.3304    0.3744       460

    accuracy                         0.4034      1279
   macro avg     0.3739    0.3712    0.3691      1279
weighted avg     0.4018    0.4034    0.3986      1279


Training weather...
Epoch 010 | val_macro_f1=0.1762 | best=0.1978
Epoch 020 | val_macro_f1=0.1198 | best=0.1978
E

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 010 | val_macro_f1=0.3725 | best=0.3943
Epoch 020 | val_macro_f1=0.3677 | best=0.3943
Epoch 030 | val_macro_f1=0.3875 | best=0.3943
Epoch 040 | val_macro_f1=0.3756 | best=0.3943
Epoch 050 | val_macro_f1=0.3877 | best=0.3943
Epoch 060 | val_macro_f1=0.3749 | best=0.3943
Epoch 070 | val_macro_f1=0.3787 | best=0.3943
Epoch 080 | val_macro_f1=0.3903 | best=0.3943
Epoch 090 | val_macro_f1=0.3743 | best=0.3943
Epoch 100 | val_macro_f1=0.3769 | best=0.3943

--- complaint TEST RESULTS ---
              precision    recall  f1-score   support

         low     0.4782    0.5889    0.5278       540
      medium     0.2410    0.2652    0.2526       279
        high     0.4528    0.3022    0.3625       460

    accuracy                         0.4152      1279
   macro avg     0.3907    0.3854    0.3809      1279
weighted avg     0.4173    0.4152    0.4083      1279


Training fusion...
Epoch 010 | val_macro_f1=0.4317 | best=0.4429
Epoch 020 | val_macro_f1=0.4380 | best=0.4438
Epoch 030 | val

In [69]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report

# Combine all features into one flat vector for XGBoost
X_all = np.concatenate([X_road, X_weather, X_complaint], axis=1)

# Same 70/15/15 split
from sklearn.model_selection import train_test_split
idx = np.arange(len(y))
idx_train, idx_temp = train_test_split(idx, test_size=0.30, stratify=y, random_state=42)
idx_val, idx_test   = train_test_split(idx_temp, test_size=0.50, stratify=y[idx_temp], random_state=42)

X_train, y_train = X_all[idx_train], y[idx_train]
X_val,   y_val   = X_all[idx_val],   y[idx_val]
X_test,  y_test  = X_all[idx_test],  y[idx_test]

# XGBoost
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50)

preds = xgb.predict(X_test)
print(classification_report(y_test, preds,
      target_names=['low', 'medium', 'high'], digits=4))
xgb_f1 = f1_score(y_test, preds, average='macro')
print(f'XGBoost macro F1: {xgb_f1:.4f}')
print(f'vs Fusion MLP:    0.4151')

[0]	validation_0-mlogloss:1.06158


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:50:48] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[50]	validation_0-mlogloss:1.02306
[100]	validation_0-mlogloss:1.02496
[150]	validation_0-mlogloss:1.03030
[200]	validation_0-mlogloss:1.03674
[250]	validation_0-mlogloss:1.04316
[300]	validation_0-mlogloss:1.04803
[350]	validation_0-mlogloss:1.05544
[400]	validation_0-mlogloss:1.06345
[450]	validation_0-mlogloss:1.07171
[499]	validation_0-mlogloss:1.08047
              precision    recall  f1-score   support

         low     0.4957    0.6352    0.5568       540
      medium     0.2083    0.0896    0.1253       279
        high     0.4518    0.4587    0.4552       460

    accuracy                         0.4527      1279
   macro avg     0.3853    0.3945    0.3791      1279
weighted avg     0.4172    0.4527    0.4262      1279

XGBoost macro F1: 0.3791
vs Fusion MLP:    0.4151
